# Chapter 14 — LoRA Fine-tuning

A pre-trained model is like a brilliant student who read the internet but never learned to converse. Supervised Fine-Tuning (SFT) teaches it to follow instructions and become a helpful assistant — the transformation from text predictor to AI chatbot.

### Glossary / Glossário

• SFT — Supervised Fine-Tuning, training a pre-trained model on labeled instruction-response pairs. / Ajuste fino supervisionado, treinar modelo pré-treinado com pares rotulados de instrução-resposta.
• LoRA — Low-Rank Adaptation, fine-tuning method that adds small trainable matrices to frozen weights. / Adaptação de baixo rank, método de fine-tuning que adiciona pequenas matrizes treináveis a pesos congelados.
• QLoRA — Quantized LoRA, combines 4-bit quantization with LoRA for memory-efficient fine-tuning. / LoRA quantizado, combina quantização de 4 bits com LoRA para fine-tuning eficiente em memória.
• fine-tuning — adapting a pre-trained model to a specific task or domain. / Adaptar modelo pré-treinado para tarefa ou domínio específico.
• rank — in LoRA, the dimension of the low-rank matrices (lower = fewer parameters). / No LoRA, dimensão das matrizes de baixo rank (menor = menos parâmetros).
• ChatML — Chat Markup Language, structured format for multi-turn conversation data. / Chat Markup Language, formato estruturado para dados de conversação multi-turno.

In [ ]:
import torch
import torch.nn as nn

print("=== LoRA (Low-Rank Adaptation) from Scratch ===\n")

d_model = 4096    # Hidden dimension - width of each layer in the transformer
rank = 16         # LoRA rank - bottleneck dimension, controls adapter capacity
lora_alpha = 32   # Scaling hyperparameter - controls adapter magnitude
scaling = lora_alpha / rank  # Effective multiplier = alpha/rank

torch.manual_seed(42)
W = nn.Linear(d_model, d_model, bias=False)  # Frozen base model weight matrix - pre-trained knowledge
W.weight.requires_grad_(False)

A = nn.Linear(d_model, rank, bias=False)   # LoRA down-projection - compresses input to low rank
B = nn.Linear(rank, d_model, bias=False)   # LoRA up-projection - expands back to model dimension, initialized to zero
nn.init.zeros_(B.weight)

base_params = d_model * d_model          # Parameter count for the full weight matrix
lora_params = 2 * d_model * rank         # Parameter count for LoRA adapters (A + B) showing efficiency
print(f"Base W:   {base_params:>12,} params (frozen)")
print(f"LoRA A+B: {lora_params:>12,} params (trainable)")
print(f"Ratio:    {lora_params/base_params*100:.2f}%")

x = torch.randn(2, 10, d_model)
with torch.no_grad():
    base_out = W(x)
    lora_out = W(x) + scaling * B(A(x))
print(f"\nDiff at init (B=0): {(base_out - lora_out).abs().mean():.6f}")

out = W(x) + scaling * B(A(x))
loss = out.sum()
loss.backward()
print(f"\nW.weight.grad:      {W.weight.grad}")
print(f"A.weight.grad norm: {A.weight.grad.norm():.4f}")
print(f"B.weight.grad norm: {B.weight.grad.norm():.4f}")

# --- Aha demo: Rank vs. trainable parameters ---
print("\n=== Aha! Rank vs. Trainable Parameters ===\n")
print(f"{'Rank':>5s} | {'LoRA Params':>12s} | {'% of Base':>9s} | {'Memory Saved':>12s}")
print("-" * 52)

for r in [4, 8, 16, 32, 64]:
    lp = 2 * d_model * r          # Trainable params for this rank
    pct = lp / base_params * 100   # Percentage of full model
    saved = (1 - lp / base_params) * 100
    print(f"{r:>5d} | {lp:>12,} | {pct:>8.2f}% | {saved:>10.2f}%")

print("\n-> Even rank=64 trains only 3% of parameters. Rank 16 is the common default.")

In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(base_model, lora_config)
# trainable: 4M / 6.7B total = 0.06%

trainer = SFTTrainer(
    model=model,
    train_dataset=chat_dataset,
    args=SFTConfig(output_dir="./sft", num_train_epochs=3, lr=2e-4),
)
trainer.train()

---

**Course:** Aprenda LLM | **Chapter 14**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.